<a href="https://colab.research.google.com/github/SabeenSaeed/machine_learning_projects/blob/main/%20work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SabeenSaeed/machine_learning_projects/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook frames the **refresh/content-opportunity scoring** lane before modeling. The starter data is pseudonymized; no client names, raw URLs, titles, or queries are used.

## 1. My lane as an ML task (type)

**Lane:** Refresh/content-opportunity scoring.

**Task type:** Ranking/scoring. The decision is not simply “will this page decline?” The practical question is **which pages should an SEO editor review first?** I would produce a priority score for every eligible page and rank pages from highest to lowest review priority. This supports a limited editorial queue better than an unranked list.

In [ ]:
task_type = "ranking/scoring"
decision_owner = "SEO editor"
action = "review and prioritize pages for refresh"
print(f"Task type: {task_type}")
print(f"Decision owner: {decision_owner}")
print(f"Supported action: {action}")

Task type: ranking/scoring
Decision owner: SEO editor
Supported action: review and prioritize pages for refresh


## 2. Target or proxy

The preferred target is a **future observed outcome**: whether a page’s search impressions decline by more than 20% in the next 30-day outcome window after the feature window. That target is appropriate for a later warehouse model because it is measured after the decision point.

For this starter-data notebook, the available target is only a **proxy**: `is_declining_proxy = (trend_direction == "down")`. It is a rule-derived label from the current comparison window, so it is useful for illustrating the target shape but must not be presented as a future causal or production target. The proxy column is never used as an input feature.

In [ ]:
import pandas as pd
from pathlib import Path

local_csv = Path("data/raw/content_refresh_anonymized.csv")
if not local_csv.exists():
    local_csv = Path("/home/ubuntu/machine_learning_projects/data/raw/content_refresh_anonymized.csv")
csv_source = str(local_csv) if local_csv.exists() else "https://raw.githubusercontent.com/SabeenSaeed/machine_learning_projects/main/data/raw/content_refresh_anonymized.csv"
starter = pd.read_csv(csv_source)
starter["is_declining_proxy"] = starter["trend_direction"].eq("down").astype("int8")
print(f"Loaded starter data from: {csv_source}")
print("Proxy column created from the starter label rule:")
print(starter["is_declining_proxy"].value_counts().sort_index().rename(index={0: "not down", 1: "down"}))
print("\nImportant: is_declining_proxy is a target/proxy, not a feature.")

Loaded starter data from: data/raw/content_refresh_anonymized.csv
Proxy column created from the starter label rule:
is_declining_proxy
not down    13738
down        16262
Name: count, dtype: int64

Important: is_declining_proxy is a target/proxy, not a feature.


## 3. Success metric

The primary success metric is **precision@K**, where K is the number of pages an editor can realistically review in one work queue. Precision@K asks: “Of the top K pages ranked for refresh, what share actually show the future decline outcome?” This matches the decision because the editor acts on the top of the queue, and false positives spend scarce editorial time. I would compare the model’s precision@K with a fixed-rule baseline at the same K. Accuracy is not the primary metric because the action is prioritized review, not equal-cost classification of every page.

In [ ]:
K = 100
print(f"Primary metric: precision@{K}")
print("Formula: relevant pages among the top K ranked pages / K")
print("Baseline comparison: evaluate the same precision@K for a fixed-rule queue")

Primary metric: precision@100
Formula: relevant pages among the top K ranked pages / K
Baseline comparison: evaluate the same precision@K for a fixed-rule queue


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content page for one client, summarized over the trailing 90-day snapshot.** The starter file is already at this page-level grain, so `content_id` identifies the page and `client_id` identifies its client. The dataframe below shows the unit directly, along with observable candidate inputs and the separate proxy target.

In [ ]:
feature_cols = [
    "impressions_90d", "clicks_90d", "sessions_90d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate"
]
context_cols = ["content_id", "client_id", "content_type", "main_intent"]
show_cols = context_cols + feature_cols + ["trend_direction", "is_declining_proxy"]
lane_slice = starter.loc[
    starter["impressions_90d"].gt(0) & starter["content_age_days"].ge(90),
    show_cols
].copy()
print(f"Starter rows loaded: {len(starter):,}")
print(f"Eligible refresh-lane rows shown: {len(lane_slice):,}")
print("One row means: one pseudonymized content page for one client at one trailing-90-day snapshot.")
display(lane_slice.head(8))
print("\nProxy rate in eligible slice:", f"{lane_slice['is_declining_proxy'].mean():.3f}")

Starter rows loaded: 30,000
Eligible refresh-lane rows shown: 30,000
One row means: one pseudonymized content page for one client at one trailing-90-day snapshot.


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,17,187,20,0.76,10.6,5.88,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,9,445,25,0.05,20.3,0.00,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,11,141,20,0.09,36.5,0.00,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,78,463,22,0.49,6.2,1.28,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,145,263,14,0.13,44.0,0.00,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,5,147,20,0.03,8.5,0.00,down,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,1,90,20,0.00,7.0,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,28,445,22,0.06,21.2,3.57,stable,0



Proxy rate in eligible slice: 0.542


## 5. Why ML beats a fixed rule here

A fixed rule such as “refresh every page with fewer than 1,200 words” is a useful baseline, but it treats one signal and one threshold as equally appropriate for every page. The refresh decision combines visibility, clicks, position, freshness, content depth, engagement, content type, and intent. Their interactions can differ across pages and clients: a high-impression page with a low CTR may deserve a different action from a low-impression page with the same word count.

ML earns a place if it improves the editor’s top-K queue over that transparent baseline on a held-out future window. It can combine several noisy signals and learn interactions while still being evaluated against the simple rule. If precision@K does not improve reliably, the fixed rule or a dashboard is the better choice. The intended claim is therefore **decision-support and directional**, not causal.

In [ ]:
assert "is_declining_proxy" not in feature_cols
assert "content_id" not in feature_cols
assert "client_id" not in feature_cols
print(f"Candidate input count: {len(feature_cols)}")
print("Candidate inputs:", ", ".join(feature_cols))
print("Excluded from inputs: content_id, client_id, trend_direction, is_declining_proxy")
print("Evaluation plan: compare model precision@K with the fixed-rule baseline on a future holdout.")

Candidate input count: 8
Candidate inputs: impressions_90d, clicks_90d, sessions_90d, content_age_days, days_since_last_update, ctr, avg_position, engagement_rate
Excluded from inputs: content_id, client_id, trend_direction, is_declining_proxy
Evaluation plan: compare model precision@K with the fixed-rule baseline on a future holdout.


## Self-check

- [x] The task type is named: ranking/scoring.
- [x] The preferred future target and the starter proxy are distinguished.
- [x] The success metric is named: precision@K.
- [x] The unit of analysis is shown as a real dataframe.
- [x] The output is tied to an SEO editor’s refresh action.
- [x] The reason ML may beat a fixed rule is explained, with a baseline comparison.
- [x] IDs and the rule-derived proxy are kept out of candidate inputs.
- [x] The notebook uses careful claims: observed, proxy, directional, and decision-support.